# MBMF Q1 — v7 runner

این notebook wrapper برای `mbmf_q1_final_v7.py` است. مدل‌ها، تست‌ها و pipeline داخل فایل Python نگه داشته شده‌اند تا notebook و script دو implementation متفاوت نشوند.

**قبل از run واقعی، rating range را از مستندات dataset صریحاً وارد کنید.**


In [ ]:
from pathlib import Path
import sys

REPO_DIR = Path('.').resolve()   # اگر notebook جای دیگری است، این مسیر را به پوشه v7 تغییر دهید
sys.path.insert(0, str(REPO_DIR))
import mbmf_q1_final_v7 as v7
print('Loaded v7 from:', v7.__file__)


## 1. تست ریاضی/نشت داده


In [ ]:
v7.synthetic_smoke_test_v7()


## 2. تنظیم experiment


In [ ]:
cfg = v7.ExperimentConfig()

# مثال MovieLens-1M:
cfg.data.data_dir = '/content/ml-1m'   # تغییر دهید
cfg.data.rating_min = 1.0
cfg.data.rating_max = 5.0
cfg.data.split_mode = 'random_per_user'

# برای run مقاله، چند outer split و چند model seed استفاده کنید.
cfg.split_seeds = (2026, 2027, 2028, 2029, 2030)
cfg.model_seeds = (11, 22, 33, 44, 55)
cfg.latent_dims = (10, 20, 40)
cfg.evaluation.ranking_ks = (10, 20)
cfg.output_dir = './results_v7'

# preset علمی پیشنهادی v7
v7.apply_q1_safe_preset(cfg)
cfg


## 3. Smoke روی dataset واقعی (اختیاری، قبل از run کامل)


In [ ]:
smoke_cfg = __import__('copy').deepcopy(cfg)
smoke_cfg.fast_debug = True
smoke_cfg.output_dir = './results_v7_smoke'
# candidates, locked = v7.run_all_experiments(smoke_cfg)


## 4. Run اصلی


In [ ]:
# candidates, locked = v7.run_all_experiments(cfg)
# locked[['model', 'test_RMSE', 'test_MAE', 'test_OOB_rate']].sort_values('test_RMSE')


## 5. Ablation suite


In [ ]:
# ablation_runs = v7.run_ablation_suite(cfg)
# ablation_runs.head()


## Method 4 در یک خط

\[
r=c\,\sigma(\operatorname{logit}(r_0^{safe}/c)+\delta),\qquad c=\sqrt{(b-a)/2}.
\]

در نتیجه هر radius از `c` کوچک‌تر است و prediction بدون clipping داخل rating range می‌ماند.
